# Notebook 2: Ego-Motion Feature Engineering
### Multimodal Driving Risk Prediction | MSc Business Analytics | Dublin Business School

---

## What This Notebook Does
Downloads frame-level vehicle telemetry (ego-motion) parquet files and aggregates them
into eight clip-level behavioural features per driving clip.
These features capture how the vehicle was driven: speed, acceleration, braking, and jerk.

## Key Steps
1. **Identify** required chunk IDs from the prepared dataset
2. **Download** ego-motion parquet files for each chunk from Hugging Face
3. **Extract** frame-level telemetry records per clip
4. **Aggregate** into eight clip-level features per clip
5. **Save** ego-motion features to Google Drive

## Features Extracted
| Feature | Description |
|---|---|
| `avg_speed` | Mean vehicle speed across all frames (m/s) |
| `max_speed` | Maximum speed recorded in the clip (m/s) |
| `avg_acceleration` | Mean acceleration (m/s²) |
| `max_acceleration` | Maximum acceleration (m/s²) |
| `avg_jerk` | Mean rate of change of acceleration (m/s³) |
| `max_jerk` | Maximum jerk, clipped at ±50 (m/s³) |
| `braking_events` | Frames with acceleration below −1.5 m/s² (value used in the code) |
| `hard_braking_events` | Frames with acceleration below −3 m/s² |

## Input
- `prepared_dataset_2000.csv`: from Notebook 1
- Ego-motion parquet files: downloaded from Hugging Face

## Output
- `ego_features_2000.csv`: 1,899 clips × 8 behavioural features

## Next Notebook
→ **Notebook 3** extracts visual features from camera frames for the same clips

> **Version 2**: an optimised, more robust version of the archived notebook in `notebooks/`. Changes are marked with `# [v2]` comments; see [`CHANGES.md`](../CHANGES.md).

> **How to run.** Edit only the **SETTINGS** cell below, then *Runtime → Run all*.
> On Colab, sign in to Hugging Face when asked (or add an `HF_TOKEN` secret) with an account that has
> accepted the NVIDIA dataset licence. Run the notebooks in order 1 → 5; each reads the previous one's output
> from `DATA_FOLDER`. No NVIDIA access? Try the 5-minute [demo](../demo/demo_pipeline.ipynb) instead.

In [ ]:
# ============================================================
# SETTINGS - the only cell you need to edit
# ============================================================
# DATA_FOLDER: where the pipeline_data folders (01_… to 05_…) live.
#   Colab  → a folder in your Google Drive (mounted automatically below)
#   Local  → ./pipeline_data next to the notebooks
import os, sys
IN_COLAB = "google.colab" in sys.modules
DATA_FOLDER = "/content/drive/MyDrive/driving-risk/pipeline_data" if IN_COLAB else os.path.abspath("pipeline_data")

if IN_COLAB and DATA_FOLDER.startswith("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")
for sub in ("01_data_preparation", "02_ego_features", "03_visual_features", "04_dataset", "05_modelling_outputs"):
    os.makedirs(os.path.join(DATA_FOLDER, sub), exist_ok=True)
base_path = DATA_FOLDER
print("Data folder:", base_path)

In [ ]:
# ============================================================
# STEP 1: INSTALL REQUIRED LIBRARIES
# ============================================================
# PURPOSE:Set up environment for ego-motion processing.
# OUTPUT:Required libraries installed with compatible versions.

# Install required libraries
!pip install -q huggingface_hub pyarrow pandas==2.2.2

# ------------------------------------------------------------
# VERIFY INSTALLATION
# ------------------------------------------------------------
import pandas as pd

print("Libraries installed successfully.\n")

print("Installed pandas version:")
print(pd.__version__)

# ------------------------------------------------------------
# FINAL CONFIRMATION
# ------------------------------------------------------------
print("\n------------------------------------------------------------")
print("Step 1 complete: Required libraries installed and verified.")
print("------------------------------------------------------------")

In [ ]:
# ============================================================
# STEP 2: AUTHENTICATE DATA ACCESS
# ============================================================
# PURPOSE:Enable access to the gated NVIDIA AV dataset.
# OUTPUT:Authenticated session for dataset download.

from huggingface_hub import login

token = os.environ.get("HF_TOKEN")              # [v2] use an HF_TOKEN secret/env var if set
if token is None and IN_COLAB:
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
    except Exception:
        token = None
login(token=token)                                # asks for a token if none is set
# ------------------------------------------------------------
# VERIFY AUTHENTICATION
# ------------------------------------------------------------
print("Login successful. Your Colab notebook is now connected to Hugging Face.")

# ------------------------------------------------------------
# FINAL CONFIRMATION
# ------------------------------------------------------------
print("\n------------------------------------------------------------")
print("Step 2 complete: Hugging Face authentication successful.")
print("------------------------------------------------------------")

In [ ]:
# ============================================================
# STEP 3: IMPORT REQUIRED LIBRARIES
# ============================================================
# PURPOSE:Load libraries for Drive access, dataset download, and ego-motion processing.
# OUTPUT:All required libraries imported and ready to use.

import os
import glob
import zipfile

import pandas as pd
import numpy as np

from huggingface_hub import hf_hub_download

# ------------------------------------------------------------
# VERIFY IMPORTS
# ------------------------------------------------------------
print("Libraries imported successfully.\n")

print("Pandas version:")
print(pd.__version__)

# ------------------------------------------------------------
# FINAL CONFIRMATION
# ------------------------------------------------------------
print("\n------------------------------------------------------------")
print("Step 3 complete: Libraries imported and environment ready.")
print("------------------------------------------------------------")

In [ ]:
# ============================================================
# STEP 3A: MOUNT GOOGLE DRIVE AND LOAD PREPARED DATASET
# ============================================================
# PURPOSE:Load the updated 2,000-clip dataset from Notebook 1.
# OUTPUT:Prepared dataset loaded for ego-motion feature extraction.

# ------------------------------------------------------------
# MOUNT GOOGLE DRIVE
# ------------------------------------------------------------
# ------------------------------------------------------------
# DEFINE INPUT FILE PATH
# ------------------------------------------------------------
# base_path comes from the SETTINGS cell at the top
input_path = f"{base_path}/01_data_preparation/prepared_dataset_2000.csv"

# ------------------------------------------------------------
# LOAD DATASET
# ------------------------------------------------------------
prepared_dataset = pd.read_csv(input_path)

# ------------------------------------------------------------
# VERIFY LOADED DATASET
# ------------------------------------------------------------
print("Prepared dataset loaded successfully.\n")

print("Input file path:")
print(input_path)

print("\nPrepared dataset shape:")
print(prepared_dataset.shape)

print("\nScenario type distribution:")
print(prepared_dataset["scenario_type"].value_counts())

print("\nUnique chunks:")
print(prepared_dataset["chunk"].nunique())

print("\nFirst 5 rows:")
display(prepared_dataset.head())

# ------------------------------------------------------------
# FINAL CONFIRMATION
# ------------------------------------------------------------
print("\n------------------------------------------------------------")
print("Step 3A complete: Updated prepared dataset loaded from Google Drive.")
print("------------------------------------------------------------")

In [ ]:
# ============================================================
# STEP 4: IDENTIFY REQUIRED EGO-MOTION CHUNKS
# ============================================================
# PURPOSE:Determine which ego-motion chunks are needed.
# OUTPUT:Unique chunk list created for efficient downloads.

# ------------------------------------------------------------
# EXTRACT UNIQUE CHUNK NUMBERS
# ------------------------------------------------------------
required_chunks = sorted(prepared_dataset["chunk"].unique())

# ------------------------------------------------------------
# PRINT CHUNK INFORMATION
# ------------------------------------------------------------
print("Required chunks identified successfully.\n")

print("Total unique chunks needed:")
print(len(required_chunks))

print("\nFirst 10 chunk IDs:")
print(required_chunks[:10])

# ------------------------------------------------------------
# FINAL CONFIRMATION
# ------------------------------------------------------------
print("\n------------------------------------------------------------")
print("Step 4 complete: Required ego-motion chunks identified.")
print("------------------------------------------------------------")

In [ ]:
# ============================================================
# STEP 5: DOWNLOAD REQUIRED EGO-MOTION CHUNKS
# ============================================================
# PURPOSE:Download only required ego-motion chunks efficiently.
# OUTPUT:Ego-motion ZIP files downloaded to Colab cache.

from huggingface_hub import hf_hub_download

repo_id = "nvidia/PhysicalAI-Autonomous-Vehicles"

# ------------------------------------------------------------
# STORE DOWNLOAD PATHS
# ------------------------------------------------------------
downloaded_chunks = []

# ------------------------------------------------------------
# DOWNLOAD LOOP
# ------------------------------------------------------------
total_chunks = len(required_chunks)

for i, chunk_id in enumerate(required_chunks):
    chunk_str = f"{int(chunk_id):04d}"

    file_path = f"labels/egomotion.offline/egomotion.offline.chunk_{chunk_str}.zip"

    local_path = hf_hub_download(
        repo_id=repo_id,
        filename=file_path,
        repo_type="dataset"
    )

    downloaded_chunks.append({
        "chunk": int(chunk_id),
        "file_path": local_path
    })

    # Progress print every 10 chunks
    if (i + 1) % 10 == 0:
        print(f"Downloaded {i + 1} / {total_chunks} chunks")

# ------------------------------------------------------------
# CONVERT TO DATAFRAME
# ------------------------------------------------------------
downloaded_chunks_df = pd.DataFrame(downloaded_chunks)

# ------------------------------------------------------------
# PRINT SUMMARY
# ------------------------------------------------------------
print("\nDownload completed successfully.\n")

print("Total chunks downloaded:")
print(len(downloaded_chunks_df))

print("\nFirst 5 entries:")
display(downloaded_chunks_df.head())

# ------------------------------------------------------------
# FINAL CONFIRMATION
# ------------------------------------------------------------
print("\n------------------------------------------------------------")
print("Step 5 complete: Ego-motion chunks downloaded efficiently.")
print("------------------------------------------------------------")

In [ ]:
# ============================================================
# STEP 6: EXTRACT EGO-MOTION FEATURES
# ============================================================
# PURPOSE:Compute stable and realistic motion features per clip.
# OUTPUT:Clean ego-motion feature dataset with minimal data loss.

import tempfile
import shutil

# ------------------------------------------------------------
# CREATE LOOKUP OF SELECTED CLIPS
# ------------------------------------------------------------
selected_clip_ids = set(prepared_dataset["clip_id"].astype(str))

# ------------------------------------------------------------
# PROCESS FUNCTION (IMPROVED)
# ------------------------------------------------------------
def process_ego_motion_file(file_path):
    df = pd.read_parquet(file_path)

    # Sort by time
    df = df.sort_values("timestamp").reset_index(drop=True)

    # Convert time
    df["time_seconds"] = df["timestamp"] / 1_000_000
    df["delta_time"] = df["time_seconds"].diff()

    # Position differences
    df["dx"] = df["x"].diff()
    df["dy"] = df["y"].diff()
    df["dz"] = df["z"].diff()

    # Distance
    df["distance"] = np.sqrt(df["dx"]**2 + df["dy"]**2 + df["dz"]**2)

    # Keep valid time intervals
    df = df[df["delta_time"] > 0.01].copy()

    # Speed
    df["speed"] = df["distance"] / df["delta_time"]

    # Remove extreme speed outliers (less aggressive)
    df = df[df["speed"] < 60].copy()

    # Acceleration
    df["acceleration"] = df["speed"].diff() / df["delta_time"]

    # Allow wider acceleration range
    df = df[
        (df["acceleration"] > -15) &
        (df["acceleration"] < 15)
    ].copy()

    # Jerk
    df["jerk"] = df["acceleration"].diff() / df["delta_time"]

    # Replace inf/nan
    df = df.replace([np.inf, -np.inf], np.nan).dropna()

    # Clamp jerk to avoid extreme spikes
    df["jerk"] = df["jerk"].clip(-50, 50)

    # Skip if too few data points
    if len(df) < 10:
        return None

    clip_id = os.path.basename(file_path).split(".")[0]

    return {
        "clip_id": clip_id,
        "avg_speed": df["speed"].mean(),
        "max_speed": df["speed"].max(),
        "avg_acceleration": df["acceleration"].mean(),
        "max_acceleration": df["acceleration"].max(),
        "avg_jerk": df["jerk"].mean(),
        "max_jerk": df["jerk"].max(),
        "braking_events": (df["acceleration"] < -1.5).sum(),
        "hard_braking_events": (df["acceleration"] < -3.0).sum()
    }

# ------------------------------------------------------------
# PROCESS ALL CHUNKS
# ------------------------------------------------------------
ego_feature_results = []

total_chunks = len(downloaded_chunks_df)

for chunk_index, row in downloaded_chunks_df.iterrows():

    chunk_id = int(row["chunk"])
    zip_path = row["file_path"]

    temp_extract_dir = tempfile.mkdtemp()

    try:
        with zipfile.ZipFile(zip_path, "r") as zip_ref:
            zip_ref.extractall(temp_extract_dir)

        parquet_files = glob.glob(os.path.join(temp_extract_dir, "*.parquet"))

        for parquet_file in parquet_files:
            clip_id = os.path.basename(parquet_file).split(".")[0]

            if clip_id in selected_clip_ids:
                try:
                    result = process_ego_motion_file(parquet_file)

                    if result is not None:
                        result["chunk"] = chunk_id
                        ego_feature_results.append(result)

                except Exception as e:
                    print("Error processing:", clip_id)

    finally:
        shutil.rmtree(temp_extract_dir)

    if (chunk_index + 1) % 10 == 0:
        print(f"Processed {chunk_index + 1} / {total_chunks} chunks")

# ------------------------------------------------------------
# CREATE FINAL DATAFRAME
# ------------------------------------------------------------
ego_features_df = pd.DataFrame(ego_feature_results)

# ------------------------------------------------------------
# PRINT RESULTS
# ------------------------------------------------------------
print("\nImproved ego-motion extraction completed.\n")

print("Ego features shape:")
print(ego_features_df.shape)

print("\nUnique clips processed:")
print(ego_features_df["clip_id"].nunique())

print("\nMissing clips:")
print(len(selected_clip_ids) - ego_features_df["clip_id"].nunique())

print("\nFirst 5 rows:")
display(ego_features_df.head())

print("\nSummary statistics:")
display(ego_features_df.describe())

# ------------------------------------------------------------
# FINAL CONFIRMATION
# ------------------------------------------------------------
print("\n------------------------------------------------------------")
print("Step 6 complete: Improved ego-motion features extracted.")
print("------------------------------------------------------------")

In [ ]:
# ============================================================
# STEP 7: MERGE EGO FEATURES WITH PREPARED DATASET
# ============================================================
# PURPOSE:Combine ego-motion features with metadata dataset.
# OUTPUT:Final dataset with behaviour features.

merged_df = prepared_dataset.merge(
    ego_features_df,
    on="clip_id",
    how="inner"
)

print("Merge completed successfully.\n")

print("Prepared dataset shape:")
print(prepared_dataset.shape)

print("\nEgo features shape:")
print(ego_features_df.shape)

print("\nMerged dataset shape:")
print(merged_df.shape)

print("\nFirst 5 rows:")
display(merged_df.head())

print("\n------------------------------------------------------------")
print("Step 7 complete: Ego-motion features merged successfully.")
print("------------------------------------------------------------")

In [ ]:
# ============================================================
# STEP 8: SAVE EGO-MOTION FEATURES TO GOOGLE DRIVE
# ============================================================
# PURPOSE:Persist Notebook 2 output for reuse in later notebooks.
# OUTPUT:CSV and Parquet ego-motion feature files saved to Google Drive.

# ------------------------------------------------------------
# CREATE NOTEBOOK 2 OUTPUT FOLDER
# ------------------------------------------------------------
# base_path comes from the SETTINGS cell at the top
notebook2_path = f"{base_path}/02_ego_features"

os.makedirs(notebook2_path, exist_ok=True)

# ------------------------------------------------------------
# DEFINE OUTPUT FILE PATHS
# ------------------------------------------------------------
csv_output_path = f"{notebook2_path}/ego_features_2000.csv"
parquet_output_path = f"{notebook2_path}/ego_features_2000.parquet"

# ------------------------------------------------------------
# SAVE FILES
# ------------------------------------------------------------
ego_features_df.to_csv(csv_output_path, index=False)
ego_features_df.to_parquet(parquet_output_path, index=False)

# ------------------------------------------------------------
# VERIFY SAVED FILES
# ------------------------------------------------------------
print("Ego-motion features saved successfully.\n")

print("CSV file saved at:")
print(csv_output_path)

print("\nParquet file saved at:")
print(parquet_output_path)

print("\nFile existence check:")
print("CSV exists:", os.path.exists(csv_output_path))
print("Parquet exists:", os.path.exists(parquet_output_path))

print("\nEgo feature dataset shape:")
print(ego_features_df.shape)

# ------------------------------------------------------------
# FINAL CONFIRMATION
# ------------------------------------------------------------
print("\n------------------------------------------------------------")
print("Step 8 complete: Ego-motion features saved to Google Drive.")
print("------------------------------------------------------------")